# Indeterminate Outcomes on TCGA

Reproduces Table `tab:app-indet` and the unimodal rates quoted in the text (Appendix, Indeterminate Outcomes on TCGA).

**Multimodal:** 50 pooled random splits × 7 models × (50 CCA variates + 50 RNA-residual PCs + 10 random labels + 53 S2B tasks) × 2 modalities, pooled over modalities.

**Unimodal FMs:** 5 FMs × 3 labels (TMB, TP53, Age) × 50 pooled splits.

In [ ]:
import json, glob
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../../data')
RESULTS_DIR = DATA_DIR / 'tcga/results'

# Load all pooled random split results (multimodal)
records = []
for f in sorted(RESULTS_DIR.glob('pooled_random_split*.json')):
    records.extend(json.load(open(f)))
print(f'Multimodal: {len(records)} records from {len(list(RESULTS_DIR.glob("pooled_random_split*.json")))} splits')

In [ ]:
MODEL_NAMES = {
    'cca_K50': 'CCA', 'clip_K50': 'CLIP', 'jive_K50': 'JIVE',
    'dssl_l0.01_b0.01_K50': 'DSSL (beta=0.01)', 'dssl_l0.1_b0.1_K50': 'DSSL (beta=0.1)',
    'dssl_l1.0_b1.0_K50': 'DSSL (beta=1.0)', 'dssl_l10.0_b10.0_K50': 'DSSL (beta=10)',
}
TASK_TYPES = {'s1': 'S1', 's2b': 'S2B', 's3': 'S3', 's4': 'S4r'}

df = pd.DataFrame({
    'model': [MODEL_NAMES[r['model']] for r in records],
    'task': [TASK_TYPES[r['task_type']] for r in records],
    'is_indeterminate': [r['predicted_scenario'] == 'indeterminate' for r in records],
})
table = (df.groupby(['model', 'task'])['is_indeterminate'].mean().mul(100)
         .unstack()[list(TASK_TYPES.values())]
         .assign(All=df.groupby('model')['is_indeterminate'].mean().mul(100))
         .reindex(list(MODEL_NAMES.values())))
print('Multimodal indeterminate rates (%):')
print(table.to_string(float_format='{:.1f}'.format))

In [ ]:
# Unimodal FM indeterminate rates (natural prevalence only)
uni_data = json.load(open(
    RESULTS_DIR.parent / 'unimodal_fm_comparison_records.json'
))

# Filter to pooled (natural prevalence) splits only
pooled = [r for r in uni_data if r['split_type'] == 'pooled']
print(f"Unimodal FM records (natural prevalence): {len(pooled)}")

uni_rows = []
for r in pooled:
    uni_rows.append({
        'FM': r['fm'].upper() if r['fm'] != 'hoptimus0' else 'H-Optimus-0',
        'label': r['label'].upper(),
        'is_indeterminate': r['verdict'] == 'Indeterminate',
    })
df_uni = pd.DataFrame(uni_rows)

# Fix FM names
fm_names = {'CONCH': 'CONCH', 'TITAN': 'TITAN', 'H-Optimus-0': 'H-Optimus-0',
            'OPENMIDNIGHT': 'OpenMidnight', 'UNI': 'UNI'}
df_uni['FM'] = df_uni['FM'].map(lambda x: fm_names.get(x, x))

# Pivot by FM × label
pivot_uni = (df_uni.groupby(['FM', 'label'])['is_indeterminate']
             .mean().mul(100).unstack(fill_value=0))

# Add overall
overall_uni = df_uni.groupby('FM')['is_indeterminate'].mean().mul(100)
pivot_uni['Overall'] = overall_uni

# Reorder
fm_order = ['CONCH', 'TITAN', 'H-Optimus-0', 'OpenMidnight', 'UNI']
pivot_uni = pivot_uni.reindex([f for f in fm_order if f in pivot_uni.index])

print("\nUnimodal FM indeterminate rates at natural prevalence (%):")
print(pivot_uni.to_string(float_format='{:.1f}'.format))
